![JohnSnowLabs](https://sparknlp.org/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/annotation/text/english/embeddings/BiEncoderMultimodalEmbeddings_OpsMM_Retrieval_Pattern.ipynb)

# Multimodal Retrieval Pattern with BiEncoderMultimodalEmbeddings (OpsMM)

This notebook shows how to build a multimodal retrieval and indexing pattern with `BiEncoderMultimodalEmbeddings`. It stops at the retriever contract: embeddings are produced and prepared for vector search, while the full RAG answer-generation step is covered in the Pinecone end-to-end RAG notebook.

The OpsMM-style bi-encoder has two towers:

- a text tower that embeds aligned document chunks
- an image tower that embeds aligned page/image regions

Both towers emit vectors in the same embedding space. Spark NLP writes them to two derived output columns:

- `mm_doc_embeddings`
- `mm_image_embeddings`

Those columns can then be indexed together or separately in a vector database.

## 1. Setup

> If you run this notebook outside the Spark NLP repo, install Spark NLP first.

In [ ]:
!pip install -q pyspark spark-nlp

In [ ]:
import os

import sparknlp
from pyspark.ml import Pipeline

from sparknlp.annotator import BiEncoderMultimodalEmbeddings, VectorDBConnector
from sparknlp.reader import LayoutAlignerForVision, ReaderAssembler

spark = sparknlp.start()
print("Spark NLP version:", sparknlp.version())
print("Apache Spark version:", spark.version)

## 2. Configuration

This example uses the sample PDF reports in the Spark NLP repository.

For vector indexing, configure your vector DB credentials before running the indexing cells. The example below uses Pinecone through `VectorDBConnector`.

In [ ]:
SAMPLE_DOCS_PATH = "examples/python/data-preprocessing/data"
OPSMM_MODEL_NAME = "ops_mm_embedding_v1_2b"
OPSMM_INDEX_NAME = "opsmm-multimodal-retrieval-demo"
OPSMM_NAMESPACE = "spark-nlp-opsmm-samples"

# Optional: Spark NLP VectorDBConnector reads provider credentials from Spark config.
# Replace with your secret management approach in real projects.
# spark.conf.set("spark.jsl.settings.vectordb.api.key", os.environ["PINECONE_API_KEY"])

pdfs = sorted(name for name in os.listdir(SAMPLE_DOCS_PATH) if name.endswith(".pdf"))
pdfs

## 3. Read multimodal PDFs

`ReaderAssembler` extracts text and image annotations from the PDF files.

`outputAsDocument=False` keeps element-level text chunks so layout metadata is preserved for alignment.

In [ ]:
reader = (
    ReaderAssembler()
    .setContentPath(SAMPLE_DOCS_PATH)
    .setContentType("application/pdf")
    .setOutputCol("reader")
    .setOutputAsDocument(False)
    .setExplodeDocs(False)
)

## 4. Align text chunks with nearby images

`LayoutAlignerForVision` creates paired outputs based on `outputCol="vision_pair"`:

- `vision_pair_doc`
- `vision_pair_image`
- `vision_pair_prompt`

For OpsMM retrieval we need the document and image columns as separate aligned inputs, so `LayoutAlignerForText` is intentionally not part of this pipeline.

In [ ]:
vision_aligner = (
    LayoutAlignerForVision()
    .setInputCols(["reader_text", "reader_image"])
    .setOutputCol("vision_pair")
    .setExplodeDocs(True)
    .setMergeImagesPerChunk(False)
    .setAddNeighborText(True)
    .setNeighborTextCharsWindow(256)
)

## 5. Embed aligned text/image pairs with OpsMM

`BiEncoderMultimodalEmbeddings` consumes one `DOCUMENT` column and one `IMAGE` column.

With `outputCol="mm"`, it creates:

- `mm_doc_embeddings` for `vision_pair_doc`
- `mm_image_embeddings` for `vision_pair_image`

In [ ]:
opsmm_embeddings = (
    BiEncoderMultimodalEmbeddings.pretrained(OPSMM_MODEL_NAME, "en")
    .setInputCols(["vision_pair_doc", "vision_pair_image"])
    .setOutputCol("mm")
    .setBatchSize(4)
)

## 6. Index text and image embeddings

The annotator emits separate text-side and image-side embedding columns, so the example uses two `VectorDBConnector` stages.

- Text indexing: `vision_pair_doc` + `mm_doc_embeddings`
- Image indexing: `vision_pair_image` + `mm_image_embeddings`

The vector DB index must already exist with the OpsMM embedding dimension.

In [ ]:
text_vector_db = (
    VectorDBConnector()
    .setInputCols(["vision_pair_doc", "mm_doc_embeddings"])
    .setOutputCol("opsmm_text_index_result")
    .setProvider("pinecone")
    .setIndexName(OPSMM_INDEX_NAME)
    .setNamespace(OPSMM_NAMESPACE)
    .setMetadataColumns(["fileName"])
    .setModalityMode("text")
    .setBatchSize(50)
)

image_vector_db = (
    VectorDBConnector()
    .setInputCols(["vision_pair_image", "mm_image_embeddings"])
    .setOutputCol("opsmm_image_index_result")
    .setProvider("pinecone")
    .setIndexName(OPSMM_INDEX_NAME)
    .setNamespace(OPSMM_NAMESPACE)
    .setMetadataColumns(["fileName"])
    .setModalityMode("image")
    .setBatchSize(50)
)

## 7. Run the indexing pipeline

`ReaderAssembler` reads from `contentPath`, so the input DataFrame can be a single empty row.

Run this cell only after your vector DB credentials and index are configured.

In [ ]:
pipeline = Pipeline(stages=[
    reader,
    vision_aligner,
    opsmm_embeddings,
    text_vector_db,
    image_vector_db,
])

empty_df = spark.createDataFrame([[]])

# Uncomment after credentials/index are configured.
# result = pipeline.fit(empty_df).transform(empty_df)
# result.select(
#     "vision_pair_doc",
#     "vision_pair_image",
#     "mm_doc_embeddings",
#     "mm_image_embeddings",
#     "opsmm_text_index_result",
#     "opsmm_image_index_result",
# ).show(5, truncate=80)

## 8. Inspect embeddings without indexing

For local debugging, you can run only the read → align → embed stages and inspect the output columns before connecting a vector DB.

In [ ]:
embedding_pipeline = Pipeline(stages=[reader, vision_aligner, opsmm_embeddings])

# This still downloads/loads the pretrained OpsMM model.
# embedded = embedding_pipeline.fit(empty_df).transform(empty_df)
# embedded.selectExpr(
#     "size(vision_pair_doc) as doc_pairs",
#     "size(vision_pair_image) as image_pairs",
#     "size(mm_doc_embeddings) as doc_vectors",
#     "size(mm_image_embeddings) as image_vectors"
# ).show(truncate=False)

## 9. Retrieval pattern

After indexing both modalities into the same namespace, downstream retrieval can use either side of the model:

- text query → OpsMM text embedding → retrieve text/image records
- image query → OpsMM image embedding → retrieve text/image records

The important contract is that text and image vectors are in the same embedding space, while metadata keeps track of `modality`, `item_id`, and document/image provenance.